
####2. Find the top 2 most delayed departure flights on 2000-01-16 from AUS to ORD

Collect the results into a list and display the output as the following.
```
    AA flight delayed by 5.0 minutes
    AA flight delayed by 2.0 minutes
    UA flight delayed by 2.0 minutes
```

In [0]:

%sql
SHOW CATALOGS

In [0]:
%sql
SELECT CURRENT_CATALOG()

In [0]:
%sql
USE CATALOG dev

In [0]:
%sql
SHOW SCHEMAS

In [0]:
%sql
USE SCHEMA spark_db

In [0]:
%sql
SHOW TABLES IN dev.spark_db;

In [0]:
flight_time = spark.read.table('dev.spark_db.flight_time')

In [0]:
flight_time.display()

find the top 3 most delayed flights from 2000-01-16 from AUS to ORD



In [0]:
# from pyspark.sql.functions import to_timestamp,to_date,col,when,expr

# flights_df =  flight_time.withColumns({
#                 "scheduled_dept_time" : to_timestamp(col('FL_DATE')) + col('CRS_DEP_TIME_IN_HR_MIN'),
#                 "Actual_dept_time" : to_timestamp(col('FL_DATE')) + col('DEP_TIME_TIME_IN_HR_MIN')
#                 })\
#                 .withColumn("Delayed_dept_time" , when( (col("Actual_dept_time") - col("scheduled_dept_time")) > expr("INTERVAL 0 SECOND") ,expr("INTERVAL ('Actual_dept_time') - ('scheduled_dept_time') DAY TO MINUTE ")).otherwise(expr("INTERVAL 0 MINUTE")))

In [0]:
from pyspark.sql.functions import to_timestamp,col

flights_df =  flight_time.withColumns({
                "scheduled_dept_time" : to_timestamp(col('FL_DATE') + col('CRS_DEP_TIME_IN_HR_MIN')),
                "Actual_dept_time" : to_timestamp(col('FL_DATE') + col('DEP_TIME_TIME_IN_HR_MIN'))})

In [0]:
from pyspark.sql.functions import unix_timestamp,desc,when


delay_in_minutes =(((unix_timestamp(col("Actual_dept_time")))- (unix_timestamp(col("scheduled_dept_time"))))/60)

flights_df_delay = flights_df.withColumn("delay_in_minutes",when(delay_in_minutes >0, delay_in_minutes).otherwise(0))

In [0]:
flights_df_delay_sorted = flights_df_delay.orderBy(desc('delay_in_minutes'))

In [0]:
flights_df_delay_sorted.display()

In [0]:
#2000-01-16 from AUS to ORD top 3 delayed flights

flights_df_delay_sorted = flights_df_delay_sorted.filter(col("FL_DATE") == "2000-01-16")

In [0]:
flights_df_delay_sorted = flights_df_delay_sorted.filter((col('ORIGIN')=="AUS") & (col("DEST")== "ORD"))

In [0]:
flights_df_delay_sorted = flights_df_delay_sorted.limit(2)

Collect the results into a list and display the output as the following.

    AA flight delayed by 5.0 minutes
    AA flight delayed by 2.0 minutes
    UA flight delayed by 2.0 minutes

In [0]:
flight_df_list = flights_df_delay_sorted.select(col('OP_CARRIER'), col('delay_in_minutes')).collect()

In [0]:
flight_df_list

In [0]:
for row in flight_df_list:
    print(f"{row.OP_CARRIER} flight delayed by {row.delay_in_minutes} minutes")